# 10.4 相同圖塊，如何知道它在左邊還是右邊？


兩塊像素內容相同的紅圖，一塊在左、一塊在右；只看內容向量無法分清所在格子。問「左邊是什麼」時，需要把位置也留下。可以像給卡片加座位提示，將等寬的位置向量加進內容向量。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

patches = torch.ones(1, 4, 3)
position = torch.arange(4.0)[None, :, None]
x = patches + position
print("形狀", tuple(x.shape))
print("位置0", x[0, 0].tolist())
print("位置3", x[0, 3].tolist())
print("仍相同", torch.equal(x[:, 0], x[:, 3]))

本例有 1 張圖、4 個位置、每位置 3 個特徵，內容全部是 1。`arange(4.0)` 產生 0、1、2、3，補軸後 `(1,4,1)`；廣播讓每位置的編號重複加到自己的三個特徵。輸出仍 `(1,4,3)`，位置 0 得 `[1,1,1]`，位置 3 得 `[4,4,4]`，相同比較為 False。

這是讓位置差異進入表示的手算式示範。正式模型可以使用可訓練位置向量，不會天然認定「加 3 就是右下」。切塊順序、位置對應與訓練要一起建立這個意義；四個位置也不能直接套在十六塊圖上。

還要分清兩種重排：只交換內容、位置提示留在原格，是把物件搬家；內容與原位置提示一起移動，只是換儲存順序，仍表示原來的空間位置。兩者不應混成同一個操作。

練習去掉 `+ position`，兩個位置都應是 `[1,1,1]`，比較變 True。它確認少掉的是位置區別，不是圖片張數或向量寬度。

<details>
<summary>回顧與查證</summary>

可回顧：[10.3每個patch的特徵](https://birdhackor.github.io/tiny-perceptron-vlm/10.3.html)、[4.1文字序列的位置](https://birdhackor.github.io/tiny-perceptron-vlm/4.1.html)。

</details>
